In [11]:
from obtain_allocation import *
from study_results_per_line import *
from generate_gaussian import *
import os
import pandas as pd
import numpy as np
import matplotlib as plt

# ANALISIS toA

In [12]:
def generate_gaussian_per_slot_all_days(dataframeinput: pd.DataFrame, param_to_gauss: str):
    """
    Analyzes 'ALL' workdays. If initial normality fails, it filters outliers
    and replaces the results with the cleaned dataset statistics.
    """

    dataframe = dataframeinput[(dataframeinput["Occupation"] <= 80) & (dataframeinput["Occupation"] >= 0)]
    dataframe = dataframe[(dataframe["DayType"] == "LA") & (dataframe["FreeDay"] == 0)]

    if dataframe.empty:
        return pd.DataFrame()

    stopnum = dataframe["Stop"].iloc[0]
    timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]

    columns = ["day", "Stop"]
    for time in timeslot:
        columns.extend([
            f"Mean_{time}",
            f"std_{time}",
            f"IsNormal_{time}",
            f"outliers_{time}",
            f"instances_{time}"
        ])

    row_data = ["ALL", stopnum]

    for time in timeslot:
        data_per_timeslot = dataframe[dataframe["TimeSlot"] == time].copy()

        gauss_hypothesis, instances_involved = is_normal_modified(
            data_per_timeslot, "ALL", time, param_to_gauss, False
        )

        outliers_found = []

        if not isinstance(instances_involved, list) and not instances_involved.empty:
            current_values = instances_involved[param_to_gauss].tolist()
            outliers_found = estimate_outliers(current_values, 30, False)

            if not gauss_hypothesis and len(outliers_found) > 0:
                # Filter out the outliers
                indices_to_drop = []
                temp_values = data_per_timeslot[param_to_gauss].tolist()
                for outlier in outliers_found:
                    if outlier in temp_values:
                        idx = data_per_timeslot.index[temp_values.index(outlier)]
                        indices_to_drop.append(idx)
                        temp_values[temp_values.index(outlier)] = None  # mark as consumed

                data_cleaned = data_per_timeslot.drop(index=indices_to_drop)

        # --- Final Stats Calculation ---
        if not isinstance(instances_involved, list) and not instances_involved.empty:
            final_values = instances_involved[param_to_gauss].tolist()

            mean_val = np.mean(final_values)
            std_val = sqrt(central_moment(final_values, 2))
            instances_list = instances_involved["Occupation"].tolist()

            row_data.extend([mean_val, std_val, gauss_hypothesis, outliers_found, instances_list])
        else:
            row_data.extend([0, 0, "NO DATA", 0, 0])

    new_dataframe = pd.DataFrame([row_data], columns=columns)
    return new_dataframe

In [13]:
files_stops = files = os.listdir("../data/E01_toA/NEW")


#Creamos plantilla para el dataframe FINAL

daytype = ["L", "M", "X", "J", "V"]
timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


#Inittialy column day to specify L, M, X, J, V
columns = ["day"]
columns.append("Stop")

#For each timeslot t add columns Mean_t, std_t and IsNormal_t
for time in timeslot:
    strvar = f"Mean_{time}"
    columns.append(strvar)
    strvar = f"std_{time}"
    columns.append(strvar)
    strvar = f"IsNormal_{time}"
    columns.append(strvar)
    strvar = f"outliers_{time}"
    columns.append(strvar)
    strvar = f"instances_{time}"
    columns.append(strvar)

#generate df
final_dataframe = pd.DataFrame(columns=columns)


# Loop over files
for file in files_stops:
    if file != "RESULTS" and file != "E01_toA.zip":
        dataframe = pd.read_csv("../data/E01_toA/NEW/" + file, encoding="utf-8", header=0)
        dataframe = clean_dataframe(dataframe, 2)
        new_dataframe = generate_gaussian_per_slot_all_days(dataframe, "Occupation")

        # Using concat as append is deprecated in newer pandas versions
        if not new_dataframe.empty:
            final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)

# ---------------------------------------------------------
# NEW: CALCULATE AND PRINT THE PERCENTAGE OF TRUE NORMALITY
# ---------------------------------------------------------
is_normal_cols = [col for col in final_dataframe.columns if "IsNormal_" in col]

# Flatten all the IsNormal columns into a single 1D array to count them easily
normality_values = final_dataframe[is_normal_cols].values.flatten()


total_true, total_tested = 0, 0

# Count how many are strictly True
total_true = (normality_values == True).sum()

total_nd = (normality_values == "NO DATA").sum()
# Count how many actually had data to test (excluding "NO DATA")
total_tested = (normality_values != "NO DATA").sum()

if total_tested > 0:
    percentage_true = (total_true / total_tested) * 100
    print(f"--- NORMALITY RESULTS ---")
    print(f"Total combinations tested: {total_tested}")
    print(f"Normal distributions (TRUE): {total_true}")
    print(f"Normal distributions (NO DATA): {total_nd}")
    print(f"Percentage of TRUE normality: {percentage_true:.2f}%")
else:
    print("No valid combinations were tested for normality.")
# ---------------------------------------------------------

with open(f"../data/E01_toA/NEW/RESULTS/FINAL_results_NORMAL.csv", "w", encoding="utf-8") as f:
        final_dataframe.to_csv(f, index=False)



/tmp/ipykernel_70430/3071965255.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)


--- NORMALITY RESULTS ---
Total combinations tested: 285
Normal distributions (TRUE): 229
Normal distributions (NO DATA): 0
Percentage of TRUE normality: 80.35%


# toB

In [14]:
files_stops = files = os.listdir("../data/E01_toB")


#Creamos plantilla para el dataframe FINAL

daytype = ["L", "M", "X", "J", "V"]
timeslot = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


#Inittialy column day to specify L, M, X, J, V
columns = ["day"]
columns.append("Stop")

#For each timeslot t add columns Mean_t, std_t and IsNormal_t
for time in timeslot:
    strvar = f"Mean_{time}"
    columns.append(strvar)
    strvar = f"std_{time}"
    columns.append(strvar)
    strvar = f"IsNormal_{time}"
    columns.append(strvar)
    strvar = f"outliers_{time}"
    columns.append(strvar)
    strvar = f"instances_{time}"
    columns.append(strvar)

#generate df
final_dataframe = pd.DataFrame(columns=columns)

# Loop over files
for file in files_stops:
    if file != "RESULTS" and file != "E01_toB.zip":
        dataframe = pd.read_csv("../data/E01_toB/" + file, encoding="utf-8", header=0)
        dataframe = clean_dataframe(dataframe, 1)
        new_dataframe = generate_gaussian_per_slot_all_days(dataframe, "Occupation")

        # Using concat as append is deprecated in newer pandas versions
        if not new_dataframe.empty:
            final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)

# ---------------------------------------------------------
# NEW: CALCULATE AND PRINT THE PERCENTAGE OF TRUE NORMALITY
# ---------------------------------------------------------
is_normal_cols = [col for col in final_dataframe.columns if "IsNormal_" in col]

# Flatten all the IsNormal columns into a single 1D array to count them easily
normality_values = final_dataframe[is_normal_cols].values.flatten()

total_true, total_tested = 0, 0

# Count how many are strictly True
total_true = (normality_values == True).sum()

total_nd = (normality_values == "NO DATA").sum()
# Count how many actually had data to test (excluding "NO DATA")
total_tested = (normality_values != "NO DATA").sum()

if total_tested > 0:
    percentage_true = (total_true / total_tested) * 100
    print(f"--- NORMALITY RESULTS ---")
    print(f"Total combinations tested: {total_tested}")
    print(f"Normal distributions (TRUE): {total_true}")
    print(f"Normal distributions (NO DATA): {total_nd}")
    print(f"Percentage of TRUE normality: {percentage_true:.2f}%")
else:
    print("No valid combinations were tested for normality.")
# ---------------------------------------------------------

with open(f"../data/E01_toB/RESULTS/FINAL_results_NORMAL.csv", "w", encoding="utf-8") as f:
        final_dataframe.to_csv(f, index=False)

/tmp/ipykernel_70430/2609048315.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe = pd.concat([final_dataframe, new_dataframe], ignore_index=True)


--- NORMALITY RESULTS ---
Total combinations tested: 218
Normal distributions (TRUE): 158
Normal distributions (NO DATA): 10
Percentage of TRUE normality: 72.48%


As the results are more than accurate for assumming the normality of the occupation value. We store, for every stop in every timeslot in both Directions, we add just the mean and std.

In [15]:
dataframe_toA = pd.read_csv("../data/E01_toA/NEW/RESULTS/FINAL_results_NORMAL.csv")

columns =["Stop"]

for time in timeslot:
    columns.append(f"Mean_{time}")
    columns.append(f"std_{time}")



final_export_dataframe_toA = dataframe_toA[columns].copy()


dataframe_toB = pd.read_csv("../data/E01_toB/RESULTS/FINAL_results_NORMAL.csv")


columns =["Stop"]

for time in timeslot:
    columns.append(f"Mean_{time}")
    columns.append(f"std_{time}")



final_export_dataframe_toB = dataframe_toB[columns].copy()


with open(f"../../WEB_DEVEL/data/known_data/NORMAL_OCCUPATION_toA.csv", "w") as f:
    final_export_dataframe_toA.to_csv(f, index=False)

with open(f"../../WEB_DEVEL/data/known_data/NORMAL_OCCUPATION_toB.csv", "w") as f:
    final_export_dataframe_toB.to_csv(f, index=False)